# Модель A: классификатор дизайна исследования (обучение в YC, plan §7)

LR-голова на замороженных эмбеддингах e5-small. Заполняет `design` там, где Publication Type молчит (`UNKNOWN`). Инференс в пайплайне — `src/adapters/classify/design_clf.py` (`ClassifierPort`).

Запуск: DataSphere → project → Run All (минуты на CPU). Метки должны совпадать со значениями `StudyDesign`: `rct`, `observational`, `review`, `case_report`.

**Чекпойнт:** `design_clf.joblib` → Object Storage → положить в `data/models/design_clf.joblib` (путь, который читает адаптер).

In [ ]:
# В DataSphere раскомментируйте при необходимости:
# %pip install -q sentence-transformers scikit-learn joblib

In [ ]:
# Демо-сэмпл (abstract, design). ЗАМЕНИТЬ на human-checked gold (50-100) из data/.
data = [
    ("Randomized controlled trial comparing drug A vs placebo, double-blind.", "rct"),
    ("We randomly assigned 200 patients to two arms in a phase III trial.", "rct"),
    ("Retrospective cohort study of registry data over 10 years.", "observational"),
    ("A prospective observational cohort followed for survival outcomes.", "observational"),
    ("This systematic review and meta-analysis pooled 12 studies.", "review"),
    ("Narrative review of immunotherapy in melanoma.", "review"),
    ("Case report of a rare adverse event in one patient.", "case_report"),
    ("We describe a single case of unexpected response.", "case_report"),
]
texts = [t for t, _ in data]
labels = [y for _, y in data]
len(texts)

In [ ]:
from sentence_transformers import SentenceTransformer

# Тот же энкодер, что и в адаптере (src/adapters/classify/design_clf.py).
encoder = SentenceTransformer("intfloat/multilingual-e5-small")
X = encoder.encode(texts, normalize_embeddings=True)
X.shape

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# Честная оценка — на hold-out, не на train (крит. 9). На крошечном демо split условный;
# на gold (50-100) берите stratified hold-out и отчитывайте per-class F1 по нему.
X_tr, X_te, y_tr, y_te = train_test_split(
    X, labels, test_size=0.25, random_state=0, stratify=labels
)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_tr, y_tr)
print(classification_report(y_te, clf.predict(X_te)))

In [ ]:
import joblib

# Переобучить на всех данных перед сохранением чекпойнта.
clf.fit(X, labels)
joblib.dump(clf, "design_clf.joblib")
# Чекпойнт -> Object Storage -> data/models/design_clf.joblib (путь адаптера).
# import boto3
# s3 = boto3.client('s3', endpoint_url='https://storage.yandexcloud.net')
# s3.upload_file('design_clf.joblib', '<bucket>', 'models/design_clf.joblib')